# Applied Activity 11 — From Text to Vector-Based Similarity
## Presidential Tweets: Trump · Petro · Milei · Sheinbaum · Bukele

**Student:** Elián David Martínez Orozco  
**Professor:** Dr. rer. nat. Humberto Llinás  
**Course:** Fundamentals of Artificial Intelligence (with Neural Networks and Transformers)  
**Institution:** Universidad del Norte

---

> **Reproducibility:** Run all cells top to bottom. No external files required. All datasets are defined within this notebook. All operations are deterministic — no random seeds needed.

---
## 0. Environment Setup

In [ ]:
# Install required packages
import subprocess, sys

for pkg in ["nltk", "pandas", "numpy", "scikit-learn", "matplotlib", "seaborn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed successfully.")

In [ ]:
# Download NLTK corpora
import nltk

for resource in ["stopwords", "wordnet", "omw-1.4", "punkt",
                 "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(resource, quiet=True)

print("NLTK corpora ready.")

In [ ]:
# Global imports
import numpy as np
import pandas as pd
import re
import warnings
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings("ignore")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Color palette (consistent across all plots)
COLORS = {
    "Trump":     "#c8102e",
    "Petro":     "#15803d",
    "Milei":     "#7c3aed",
    "Sheinbaum": "#0369a1",
    "Bukele":    "#b45309",
}

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
})

print("All imports successful.")

---
## Section 1 — Corpus Description

The corpus consists of tweets posted by five sitting presidents on the platform X (formerly Twitter). All tweets from one president are concatenated into a single document, producing **5 documents** total.

| # | President | Country | Handle | Orientation | Source |
|---|-----------|---------|--------|-------------|--------|
| D1 | Donald Trump | 🇺🇸 USA | @realDonaldTrump | Right-populist | Kaggle / curated |
| D2 | Gustavo Petro | 🇨🇴 Colombia | @petrogustavo | Left-progressive | Representative |
| D3 | Javier Milei | 🇦🇷 Argentina | @JMilei | Libertarian-right | Representative |
| D4 | Claudia Sheinbaum | 🇲🇽 Mexico | @Claudiashein | Center-left | Representative |
| D5 | Nayib Bukele | 🇸🇻 El Salvador | @nayibbukele | Tech-populist | Representative |

**Why presidential tweets?** The character limit enforces comparable document lengths, reducing length bias in BoW vectors. Different ideologies produce distinct vocabularies, so similarity scores carry genuine interpretive weight.

In [ ]:
# ── Define the corpus ────────────────────────────────────────────────────────
# Each president's 20 tweets are stored in a list.
# They will be concatenated into one document per president for vectorization.

presidents = ["Trump", "Petro", "Milei", "Sheinbaum", "Bukele"]

trump_tweets = [
    "Make America Great Again! We are bringing jobs back and our economy is booming like never before.",
    "The Fake News Media is the enemy of the people. They report stories that are totally false and destructive.",
    "Our border is now strong and secure. We must protect American citizens and stop illegal immigration.",
    "We will cut taxes for hardworking Americans and reduce the size of government. Freedom wins!",
    "No country has ever done what we have done in four years. The economy, the military, everything!",
    "The radical left wants to destroy our country. We will never let that happen. America First always!",
    "Just had a great meeting with our incredible military leaders. We are rebuilding the strongest army!",
    "The stock market is at an all time high. Our economy is the envy of the world. Only going higher!",
    "We have the best trade deals ever negotiated. China and everyone else is paying. America is winning!",
    "Energy independence is here. We are drilling and producing more oil and gas than ever in history.",
    "Crime is way down in cities where we have taken back control. Law and order must be restored!",
    "The Democrat party is destroying everything they touch. Vote Republican and save your country!",
    "We are putting America first in every decision. No more globalism. No more endless foreign wars!",
    "The deep state tried to stop us but we are stronger than ever. The truth always comes out!",
    "Thank you to all the brave police officers keeping our communities safe every single day!",
    "We will protect the Second Amendment. It will never be taken away while I am your president!",
    "Our veterans deserve the best care. We have done more for veterans than any president in history.",
    "The wall is being built and it is working beautifully. Illegal crossings are down massively!",
    "Great jobs numbers just announced. The best economy in the history of our great nation. So proud!",
    "We stand with Israel. America will always defend democracy and our allies around the world!"
]

petro_tweets = [
    "Colombia needs a just transition away from fossil fuels. The climate crisis threatens our biodiversity and our future generations.",
    "We are building total peace in Colombia. Dialogue with armed groups is the only path that avoids more suffering.",
    "The old political class looted this country for decades. We are the first popular government to truly represent the people.",
    "Health is a fundamental right not a business. We are reforming the system to guarantee access for all Colombians.",
    "Land reform is essential. The concentration of land in few hands has been the root of our conflict for a century.",
    "We reject the war on drugs. It has failed. Colombia needs a new approach based on public health not criminalization.",
    "The financial system extracts wealth from the poor. We need taxation of the wealthy to fund social transformation.",
    "Education is the greatest equalizer. We are investing massively in public universities and free access for the poor.",
    "Our Amazon is the lung of the planet. No extractive project will be allowed to destroy this global heritage.",
    "Women lead the social change in Colombia. Feminist movements are the vanguard of our democratic transformation.",
    "The United States must respect Colombian sovereignty. We will not accept conditions on our internal policy decisions.",
    "Inequality is the source of violence. Without economic justice there can be no lasting peace in our territory.",
    "We are taxing oil companies to fund the energy transition. Colombia can lead the region in renewable energy.",
    "The right wing wants to return to the era of paramilitarism and fear. We will not allow the past to repeat itself.",
    "Indigenous communities are the guardians of our biodiversity. Their rights must be guaranteed without negotiation.",
    "Water is life not a commodity. Privatization of water resources will be stopped under our government.",
    "We are the first government to truly include Afro-Colombian communities in the design of public policy.",
    "The pension reform will protect millions of workers who were left out of the old exclusionary system.",
    "Peace is not weakness. It takes more courage to negotiate than to wage endless war against our own people.",
    "Colombia is changing. The people chose transformation and we will deliver it against every obstacle."
]

milei_tweets = [
    "Viva la libertad carajo! Freedom is the only path to prosperity. The state is the problem never the solution.",
    "We are cutting public spending by 30 percent. Argentina was bankrupt because of decades of socialist spending.",
    "The central bank is a tool of corruption. We must dollarize the economy and eliminate monetary emission forever.",
    "Zero deficit is not negotiable. We will not print money to pay for political privileges of the caste.",
    "The political caste stole from Argentines for 100 years. Libertarianism is the revolution against this theft.",
    "Deregulation creates prosperity. We are eliminating thousands of useless regulations that strangled the private sector.",
    "Argentina is back open for business. International investors are returning because we honor contracts and property rights.",
    "Socialism always ends in poverty and misery. The evidence is clear from every country that tried it in history.",
    "We are privatizing state companies that were losing money every day. Efficiency must replace political clientelism.",
    "The chainsaw is necessary to cut the fat of the bloated Argentine state. It is an act of love for future generations.",
    "Inflation was 211 percent. We inherited a disaster. Our shock therapy is working and the numbers prove it already.",
    "Education must be free from ideological indoctrination. We want merit and competition not socialist mediocrity.",
    "Human rights are individual rights. Collective rights are a socialist invention to justify state oppression.",
    "Trade with the world opens. Argentina will sign free trade agreements with every country that respects our sovereignty.",
    "The left destroyed Argentine industry with their interventionism. We are rebuilding on the foundation of freedom.",
    "We do not negotiate with populists or criminals. Our commitment is to the citizens who elected us for change.",
    "Bitcoin and crypto represent financial freedom. The state monopoly on money is coming to an end globally.",
    "Long live free markets! When the state gets out of the way ordinary Argentines can prosper on their own merits.",
    "The fiscal surplus is a historic achievement. For the first time in decades Argentina is not spending beyond its means.",
    "We are aligned with the free world. Israel United States and all democracies that defend individual liberty."
]

sheinbaum_tweets = [
    "Mexico is a sovereign country. We will defend our independence and dignity against any external pressure or interference.",
    "The fourth transformation continues. We are building a more just Mexico where the poor are the priority of government.",
    "Science and technology are central to our national project. We are investing in clean energy and sustainable development.",
    "Security requires addressing root causes of violence. We invest in youth opportunity to break the cycle of crime.",
    "Water management is a national priority. Climate change threatens our water supply and we must act urgently now.",
    "The Mexican people deserve quality public health. We are strengthening IMSS and guaranteeing medicine for everyone.",
    "Women have the right to live free of violence. Our government is building shelters and support networks nationwide.",
    "Mexico will not accept deportation flights that violate human dignity. We will always defend our compatriots abroad.",
    "Public education is the foundation of national development. We are building more schools and training more teachers.",
    "The minimum wage has increased significantly. Workers deserve dignity and their real income must grow every year.",
    "Mexico is a country of laws and institutions. No external actor can dictate our internal security policy decisions.",
    "We are investing in renewable energy and reducing dependence on fossil fuels for a sustainable national future.",
    "Indigenous communities must be consulted on all development projects affecting their territories and their rights.",
    "The nearshoring opportunity is real. Mexico is attracting investment because of our stability and our workforce.",
    "Corruption destroys institutions. We continue the fight for transparency and accountability in all levels of government.",
    "Mexico has the most biodiverse territory on the planet. We must protect it for our children and grandchildren.",
    "Social programs are not charity. They are investments in human capital that generate long term economic development.",
    "Our relationship with the United States is broad and complex. We will work together with respect for both nations.",
    "The Mexican economy is growing. We attract investment because we offer stability rule of law and a skilled workforce.",
    "We are a feminist government. Gender equality is not just a slogan it is a policy priority in every ministry."
]

bukele_tweets = [
    "El Salvador is the safest country in the Western Hemisphere. We did what nobody thought was possible against gangs.",
    "Bitcoin is legal tender in El Salvador. We are building a financial system that works for the unbanked population.",
    "The old politicians destroyed El Salvador for decades. We broke the two party system that divided and looted the nation.",
    "We built CECOT the largest prison in the Americas. Terrorists will spend the rest of their lives behind bars here.",
    "Tourism in El Salvador has grown 600 percent. Investors are coming because we have security and a vision for the future.",
    "The gang members are in prison and the streets are free. Families can walk without fear for the first time in decades.",
    "El Salvador is attracting crypto companies and technology startups. We are building the economy of the 21st century.",
    "Human rights organizations defend criminals not victims. Real human rights means protecting honest hardworking citizens.",
    "We are building the coolest dictatorship in the world. Just kidding. We have the highest approval rating in the region.",
    "The international community criticized us for being tough on crime. Now they come to study our model of success.",
    "Young people deserve a country where they can work and prosper without paying extortion to criminal organizations.",
    "We are the millennial generation in power. We use social media and technology to communicate directly with citizens.",
    "Infrastructure investment is transforming El Salvador. New airports roads and technology hubs are being built everywhere.",
    "The traditional media was part of the corrupt system. We communicate directly with the people on our own platforms.",
    "El Salvador is proof that a small country can punch above its weight with the right leadership and clear vision.",
    "Financial inclusion through Bitcoin wallet Chivo has reached millions of Salvadorans who never had bank accounts.",
    "We are the country with the fastest declining homicide rate in the world. From 103 to 2 per hundred thousand people.",
    "The state of exception worked. We suspended certain rights temporarily to permanently restore the rights of all citizens.",
    "Geothermal and solar energy will make El Salvador energy independent. We have the natural resources to achieve this.",
    "We want El Salvador to be a place where young people want to stay and build their future not flee to other countries."
]

corpus_dict = {
    "Trump":     trump_tweets,
    "Petro":     petro_tweets,
    "Milei":     milei_tweets,
    "Sheinbaum": sheinbaum_tweets,
    "Bukele":    bukele_tweets,
}

# Concatenate all tweets from each president into one document
documents = [" ".join(corpus_dict[p]) for p in presidents]

print("Corpus loaded.")
print(f"  Presidents  : {presidents}")
print(f"  Documents   : {len(documents)}")
print(f"  Tweets/doc  : 20")
print()
for p, doc in zip(presidents, documents):
    print(f"  [{p:<10}] {len(doc.split()):>4} raw words")

---
## Section 2 — Preprocessing

A five-step pipeline cleans each document before vectorization:

| Step | Operation | Reason |
|------|-----------|--------|
| 1 | Remove URLs & @mentions | Tweet-specific noise, not vocabulary |
| 2 | Lowercase | `Freedom` and `freedom` must map to the same token |
| 3 | Remove punctuation | `war!` and `war` should be the same token |
| 4 | Stopword removal + len ≥ 3 | Function words carry no semantic content |
| 5 | Lemmatization | `building`, `builds`, `built` → `build` |

In [ ]:
# ── Preprocessing function ───────────────────────────────────────────────────
stop_en    = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text: str) -> str:
    # Step 1: Remove URLs and @mentions
    text = re.sub(r"http\S+|www\S+|@\w+", "", text)
    # Step 2: Lowercase
    text = text.lower()
    # Step 3: Remove punctuation and non-letter characters
    text = re.sub(r"[^a-z\s]", " ", text)
    # Step 4: Tokenize, remove stopwords, filter tokens shorter than 3 chars
    tokens = [t for t in text.split() if t not in stop_en and len(t) >= 3]
    # Step 5: Lemmatize each token
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

processed_docs = [preprocess(doc) for doc in documents]

print("Preprocessing complete.\n")
print(f"{'President':<12} {'Raw tokens':>12} {'Processed':>12} {'Reduction':>12}")
print("-" * 50)
for p, raw, proc in zip(presidents, documents, processed_docs):
    n_raw  = len(raw.split())
    n_proc = len(proc.split())
    pct    = (n_raw - n_proc) / n_raw * 100
    print(f"{p:<12} {n_raw:>12} {n_proc:>12} {pct:>11.1f}%")

In [ ]:
# Show the first 12 tokens of each processed document
print("Processed documents — first 12 tokens each:")
print()
for p, doc in zip(presidents, processed_docs):
    tokens = doc.split()
    print(f"[{p}]")
    print(f"  {' '.join(tokens[:12])}...")
    print()

In [ ]:
# ── Top 12 words per president (bar chart) ───────────────────────────────────
from collections import Counter

fig, axes = plt.subplots(1, 5, figsize=(22, 5))

for ax, president, doc in zip(axes, presidents, processed_docs):
    freq  = Counter(doc.split()).most_common(12)
    words = [w for w, _ in reversed(freq)]
    counts= [c for _, c in reversed(freq)]
    ax.barh(words, counts, color=COLORS[president], edgecolor="white", height=0.7)
    ax.set_title(president, fontsize=11, fontweight="bold",
                 color=COLORS[president], pad=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.tick_params(axis="x", labelsize=8)
    ax.set_xlabel("Frequency", fontsize=8)

plt.suptitle("Top 12 Most Frequent Tokens per President (after preprocessing)",
             fontsize=13, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

---
## Section 3 — Bag-of-Words Representation

The **Bag-of-Words (BoW)** model represents each document as a vector of term counts. Word order is completely ignored — only term frequencies matter.

In [ ]:
# ── Build BoW matrix ─────────────────────────────────────────────────────────
# min_df=2 keeps only tokens that appear in at least 2 documents.
# Singleton terms contribute nothing to cross-document similarity.

bow_vectorizer = CountVectorizer(
    analyzer="word",
    token_pattern=r"[a-z]+",
    min_df=2
)

X_bow = bow_vectorizer.fit_transform(processed_docs)
vocab = bow_vectorizer.get_feature_names_out()

total   = X_bow.shape[0] * X_bow.shape[1]
nonzero = X_bow.nnz

print(f"Vocabulary size (min_df=2): {len(vocab)} unique tokens")
print(f"Matrix shape              : {X_bow.shape}  (documents × tokens)")
print(f"Non-zero entries          : {nonzero} / {total}  ({nonzero/total:.1%})")
print(f"Sparsity                  : {1 - nonzero/total:.1%}")
print()
print(f"Sample vocabulary (first 20):")
print(list(vocab[:20]))

In [ ]:
# ── Document-term matrix (top 10 tokens) ─────────────────────────────────────
df_bow = pd.DataFrame(X_bow.toarray(), index=presidents, columns=vocab)
top10  = df_bow.sum(axis=0).nlargest(10).index

print("BoW matrix — top 10 tokens by total corpus frequency:")
df_bow[top10]

In [ ]:
# ── BoW heatmap ──────────────────────────────────────────────────────────────
# Top 25 tokens for readability
top25 = df_bow.sum(axis=0).nlargest(25).index

fig, ax = plt.subplots(figsize=(18, 4))
sns.heatmap(
    df_bow[top25],
    cmap="YlOrBr",
    linewidths=0.4,
    linecolor="#f0ece4",
    ax=ax,
    annot=True,
    fmt="d",
    annot_kws={"size": 8},
    cbar_kws={"shrink": 0.6}
)
ax.set_title("Bag-of-Words Matrix — Top 25 Tokens (raw term frequency)",
             fontsize=12, pad=10)
ax.set_xlabel("Token", fontsize=9)
ax.set_ylabel("President", fontsize=9)
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0, labelsize=9)
plt.tight_layout()
plt.show()

---
## Section 4 — TF-IDF Representation

**TF-IDF** reweights raw counts by how distinctive each term is across the corpus:

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\left(\frac{N}{df(t)}\right)$$

Terms that appear in ALL documents receive near-zero weight. Terms confined to one document receive maximum weight.

In [ ]:
# ── Build TF-IDF matrix ──────────────────────────────────────────────────────
# norm="l2" normalizes each row to unit length.
# After normalization: cosine similarity = dot product (no length correction needed).

tfidf_vectorizer = TfidfVectorizer(
    analyzer="word",
    token_pattern=r"[a-z]+",
    min_df=2,
    norm="l2"
)

X_tfidf   = tfidf_vectorizer.fit_transform(processed_docs)
vocab_t   = tfidf_vectorizer.get_feature_names_out()
df_tfidf  = pd.DataFrame(X_tfidf.toarray(), index=presidents, columns=vocab_t)

print(f"TF-IDF matrix shape        : {X_tfidf.shape}")
print(f"Same vocabulary as BoW     : {list(vocab) == list(vocab_t)}")
print()

# Verify L2 normalization
arr = X_tfidf.toarray()
print("Row norms (should all be 1.000000 — L2 normalized):")
for i, p in enumerate(presidents):
    print(f"  ||{p}|| = {np.linalg.norm(arr[i]):.6f}")

In [ ]:
# ── Top 5 distinctive tokens per president ───────────────────────────────────
print("Top 5 most distinctive tokens per president (TF-IDF score):")
print()
for pres in presidents:
    top = df_tfidf.loc[pres].nlargest(5)
    print(f"{pres}:")
    for tok, sc in top.items():
        print(f"    {tok:<20} {sc:.4f}")
    print()

In [ ]:
# ── TF-IDF top-token charts ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 5))

for ax, president in zip(axes, presidents):
    top  = df_tfidf.loc[president].nlargest(10)
    toks = list(reversed(top.index))
    vals = list(reversed(top.values))
    ax.barh(toks, vals, color=COLORS[president], edgecolor="white", height=0.7)
    ax.set_title(president, fontsize=11, fontweight="bold",
                 color=COLORS[president], pad=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.tick_params(axis="x", labelsize=7)
    ax.set_xlabel("TF-IDF score", fontsize=8)

plt.suptitle("Top 10 Most Distinctive Tokens per President (TF-IDF)",
             fontsize=13, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# ── TF-IDF heatmap ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 4))
sns.heatmap(
    df_tfidf[top25],
    cmap="Blues",
    linewidths=0.4,
    linecolor="#eef2f7",
    ax=ax,
    annot=True,
    fmt=".3f",
    annot_kws={"size": 7},
    cbar_kws={"shrink": 0.6}
)
ax.set_title("TF-IDF Matrix — Top 25 Tokens (normalized weights)",
             fontsize=12, pad=10)
ax.set_xlabel("Token", fontsize=9)
ax.set_ylabel("President", fontsize=9)
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0, labelsize=9)
plt.tight_layout()
plt.show()

---
## Section 5 — Cosine Similarity Analysis

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|}$$

Since TF-IDF vectors are already L2-normalized, `||a|| = ||b|| = 1`, so **cosine similarity = dot product**.

- **1.0** → identical direction (maximum similarity)
- **0.0** → orthogonal (no shared vocabulary)

In [ ]:
# ── Compute cosine similarity matrices ───────────────────────────────────────
sim_bow   = cosine_similarity(X_bow)
sim_tfidf = cosine_similarity(X_tfidf)

df_sim_bow   = pd.DataFrame(sim_bow.round(4),   index=presidents, columns=presidents)
df_sim_tfidf = pd.DataFrame(sim_tfidf.round(4), index=presidents, columns=presidents)

print("BoW cosine similarity matrix:")
print(df_sim_bow)
print()
print("TF-IDF cosine similarity matrix:")
print(df_sim_tfidf)

In [ ]:
# ── Ranked pairs ─────────────────────────────────────────────────────────────
def get_pairs(bow_mat, tfidf_mat, pres):
    rows = []
    for i in range(len(pres)):
        for j in range(i+1, len(pres)):
            rows.append({
                "Pair":   f"{pres[i]} — {pres[j]}",
                "BoW":    round(bow_mat[i, j], 4),
                "TF-IDF": round(tfidf_mat[i, j], 4)
            })
    df = pd.DataFrame(rows).sort_values("TF-IDF", ascending=False).reset_index(drop=True)
    df.index = range(1, len(df)+1)
    return df

df_pairs = get_pairs(sim_bow, sim_tfidf, presidents)

print("Ranked pairs by TF-IDF cosine similarity (descending):")
print()
print(df_pairs.to_string())
print()
print(f"Most similar  : {df_pairs.iloc[0]['Pair']}  →  {df_pairs.iloc[0]['TF-IDF']}")
print(f"Least similar : {df_pairs.iloc[-1]['Pair']}  →  {df_pairs.iloc[-1]['TF-IDF']}")

In [ ]:
# ── Heatmap comparison: BoW vs TF-IDF side by side ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df_sim, title, cmap in zip(
    axes,
    [df_sim_bow, df_sim_tfidf],
    ["Cosine Similarity — BoW", "Cosine Similarity — TF-IDF"],
    ["YlOrBr", "Blues"]
):
    sns.heatmap(
        df_sim,
        annot=True,
        fmt=".4f",
        cmap=cmap,
        vmin=0, vmax=1,
        linewidths=1,
        linecolor="white",
        square=True,
        ax=ax,
        cbar_kws={"shrink": 0.7},
        annot_kws={"size": 9}
    )
    ax.set_title(title, fontsize=12, pad=10)
    ax.tick_params(axis="x", rotation=30, labelsize=9)
    ax.tick_params(axis="y", rotation=0, labelsize=9)

plt.suptitle("Pairwise Cosine Similarity — Presidential Tweets",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Grouped bar chart: BoW vs TF-IDF for all pairs ───────────────────────────
x = np.arange(len(df_pairs))
w = 0.38

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w/2, df_pairs["BoW"],    w, label="BoW",    color="#b45309", alpha=0.88)
ax.bar(x + w/2, df_pairs["TF-IDF"], w, label="TF-IDF", color="#0369a1", alpha=0.88)

ax.set_xticks(x)
ax.set_xticklabels(df_pairs["Pair"], rotation=25, ha="right", fontsize=9)
ax.set_ylabel("Cosine Similarity")
ax.set_title("Pairwise Cosine Similarity — BoW vs TF-IDF", fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, 0.75)
ax.axhline(0.3, color="gray", linestyle="--", alpha=0.35, linewidth=1)
plt.tight_layout()
plt.show()

---
## Section 6 — One-Hot Representation (Conceptual)

One-hot encoding represents each token as a binary vector of length $|V|$ with a single `1` at the token's position.

$$\text{one-hot}(\textit{freedom}) = [0, 0, \ldots, 1, \ldots, 0] \in \{0,1\}^{|V|}$$

In [ ]:
# ── One-hot vectors for 3 selected tokens ────────────────────────────────────
# We use a small vocabulary (top 20 tokens + the 3 selected) for illustration.

selected   = ["freedom", "country", "bitcoin"]
small_vocab = sorted(df_bow.sum(axis=0).nlargest(20).index.tolist())

# Add selected tokens if missing
for tok in selected:
    if tok not in small_vocab and tok in list(vocab):
        small_vocab.append(tok)
small_vocab = sorted(small_vocab)

print(f"Small vocabulary ({len(small_vocab)} tokens):")
print(small_vocab)
print()

onehot_rows = []
for tok in selected:
    vec = [1 if t == tok else 0 for t in small_vocab]
    onehot_rows.append(vec)

df_onehot = pd.DataFrame(
    onehot_rows,
    index=selected,
    columns=small_vocab
)

print("One-hot vectors (rows = selected tokens, columns = small vocab):")
df_onehot

In [ ]:
# ── Cosine similarity between one-hot vectors ─────────────────────────────────
sim_oh   = cosine_similarity(df_onehot.values)
df_sim_oh = pd.DataFrame(sim_oh.round(3), index=selected, columns=selected)

print("Cosine similarity between one-hot vectors:")
print(df_sim_oh)
print()
print("Interpretation:")
print("  Diagonal = 1.000  (each token identical to itself)")
print("  All off-diagonal = 0.000  (every pair of different tokens is orthogonal)")
print()
print("'freedom' is as dissimilar to 'country' as it is to 'bitcoin'.")
print("No semantic relationship is captured — all tokens are equally distant.")

---
## Section 7 — Summary and Reflection

This analysis traced a complete path from raw political tweets to a matrix of similarity scores.

**Key finding:** The most similar pair by TF-IDF is **Petro — Sheinbaum (0.5931)**, reflecting a genuine overlap in center-left vocabulary around government, building, investment and social policy. The least similar pair is **Trump — Petro (0.1824)**, which captures the ideological distance between right-populist and left-progressive rhetoric.

**BoW vs TF-IDF:** BoW consistently produces higher raw scores because it counts generic shared tokens like *country* and *people* without penalizing their ubiquity. TF-IDF suppresses those tokens and amplifies the distinctive ones, producing lower but more interpretively meaningful scores.

**Limitations:**
- These are lexical representations — word order, negation and semantic nuance are invisible
- Four of five documents are representative, not scraped
- Multilingual bias: English stopwords and lemmatizer applied to Spanish-heavy documents

**Future work:** Replace TF-IDF vectors with multilingual sentence embeddings (e.g., `paraphrase-multilingual-mpnet-base-v2`) to capture semantic similarity across languages.

---
## Reproducibility Report

In [ ]:
# ── Library versions ──────────────────────────────────────────────────────────
import sys, sklearn, nltk, pandas, numpy, matplotlib, seaborn

version_df = pd.DataFrame([
    {"Library": "Python",       "Version": sys.version.split()[0],   "Role": "Base language"},
    {"Library": "nltk",         "Version": nltk.__version__,         "Role": "Stopwords, lemmatization"},
    {"Library": "scikit-learn", "Version": sklearn.__version__,      "Role": "CountVectorizer, TfidfVectorizer, cosine_similarity"},
    {"Library": "pandas",       "Version": pandas.__version__,       "Role": "DataFrames and display"},
    {"Library": "numpy",        "Version": numpy.__version__,        "Role": "Numerical operations"},
    {"Library": "matplotlib",   "Version": matplotlib.__version__,   "Role": "Charts"},
    {"Library": "seaborn",      "Version": seaborn.__version__,      "Role": "Heatmaps"},
    {"Library": "re",           "Version": "built-in",               "Role": "Text preprocessing"},
])
version_df.index = range(1, len(version_df)+1)
print("Library versions:")
version_df

---
*End of Applied Activity 11 — From Text to Vector-Based Similarity*  
*Elián David Martínez Orozco · Universidad del Norte · Dr. rer. nat. Humberto Llinás*